# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RAJkumar5736602/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Rule:** score = 0.35·age percentile + 0.25·staleness percentile + 0.25·visibility percentile + 0.15·[position 4-20]. Reason codes: old_page_striking_distance, old_page, striking_distance, stale_update, monitor.

In [1]:
import sys, pathlib
root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p/'data/raw').exists())
sys.path.insert(0, str(root/'work/scripts'))
from wk import *
import pandas as pd, numpy as np
df = load()
import inspect; print(inspect.getsource(baseline_score))

def baseline_score(df):
    """My transparent rule: older, stale, visible pages that sit just off the top get reviewed first."""
    pos = pd.to_numeric(df["avg_position"], errors="coerce").fillna(100)
    s = (0.35 * (df["content_age_days"].rank(pct=True))
         + 0.25 * (df["days_since_last_update"].rank(pct=True))
         + 0.25 * np.log1p(df["impressions_90d"]).rank(pct=True)
         + 0.15 * pos.between(4, 20).astype(float))
    reasons = np.select([(df["content_age_days"] > 365) & pos.between(4, 20), df["content_age_days"] > 365,
                         pos.between(4, 20), df["days_since_last_update"] > 90],
                        ["old_page_striking_distance", "old_page", "striking_distance", "stale_update"], "monitor")
    return s, reasons



## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Score, rank, write `work/outputs/baseline_action_score.csv` (gitignored by design — CSVs never committed).

In [2]:
s, r = baseline_score(df)
q = pd.DataFrame({'content_id':df.content_id,'score':s.round(4),'reason':r}).sort_values('score',ascending=False)
q.to_csv(OUT/'baseline_action_score.csv', index=False)
tr, te = client_split(df)
res = evaluate(df.y.values[te], s.values[te]); print('baseline on client-holdout test:', res)
import json; json.dump(res, open(OUT/'baseline_metrics.json','w'), indent=1)

baseline on client-holdout test: {'roc_auc': 0.463, 'avg_precision': 0.47, 'p@50': 0.36, 'p@200': 0.325, 'base_rate': 0.511, 'n': 6163}


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Top-20 with reason, and whether it actually declined (for review only).

In [3]:
top = df.loc[q.index[:20], ['content_age_days','days_since_last_update','impressions_90d','avg_position','trend_direction']].assign(reason=q.reason[:20].values)
top['action'] = np.where(top.reason.str.contains('striking'), 'refresh + on-page review', 'refresh')
top

,content_age_days,days_since_last_update,impressions_90d,avg_position,trend_direction,reason,action
6653,537,104,517715,4.2,down,old_page_striking_distance,refresh + on-page review
18853,537,104,48413,4.7,down,old_page_striking_distance,refresh + on-page review
17103,537,104,37534,7.1,down,old_page_striking_distance,refresh + on-page review
14797,537,104,30384,5.1,stable,old_page_striking_distance,refresh + on-page review
15017,504,104,52681,4.5,stable,old_page_striking_distance,refresh + on-page review
148,504,104,41650,4.4,down,old_page_striking_distance,refresh + on-page review
9415,537,104,19140,4.9,stable,old_page_striking_distance,refresh + on-page review
7668,504,104,28061,6.7,down,old_page_striking_distance,refresh + on-page review
9851,504,104,27348,9.1,down,old_page_striking_distance,refresh + on-page review
4590,537,104,18167,6.9,down,old_page_striking_distance,refresh + on-page review


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks: old high-impression pages that were actually *up* — age alone over-flags. No label-derived or 30-day window fields are used in the rule (asserted below).

In [4]:
print('top-20 not declining:', int((top.trend_direction!='down').sum()))
import inspect; src = inspect.getsource(baseline_score)
assert 'trend' not in src and 'last_30' not in src and 'prev_30' not in src; print('no leakage fields in rule')

top-20 not declining: 8
no leakage fields in rule


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.